In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'b_prepare'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# image imports
from itertools import product
from PIL import Image
import torch.nn.functional as F
images = torch.stack([torch.from_numpy(np.asarray(Image.open(INPUT / 'assets' / f'{name}.png')).copy()).float() / 255
                      for name in ['cat', 'smiley']])
rng = np.random.Generator(np.random.PCG64(CFG['seed']).jumped(100))

In [ ]:
# transform renderer; CPU only, target construction has no backward pass
def render(images, angle, dx, dy, saturation, gains):
    size = images.shape[1]
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing='ij')
    center = size // 2
    x, y = xx.float() - center - dx, -(yy.float() - center - dy)
    theta = np.deg2rad(angle)
    sx = center + np.cos(theta) * x + np.sin(theta) * y
    sy = center + np.sin(theta) * x - np.cos(theta) * y
    grid = torch.stack([2 * (sx + 0.5) / size - 1, 2 * (sy + 0.5) / size - 1], -1)
    rgb, alpha = images[..., :3], images[..., 3:4]
    luminance = (rgb * torch.tensor([0.2126, 0.7152, 0.0722])).sum(-1, keepdim=True)
    color = (luminance + saturation * (rgb - luminance)) * torch.tensor(gains)
    rgba = torch.cat([color.clamp(0, 1) * alpha, alpha], -1)
    target = F.grid_sample(rgba.permute(0, 3, 1, 2), grid[None].expand(2, -1, -1, -1),
                           mode='bilinear', padding_mode='zeros', align_corners=False)
    return target.permute(0, 2, 3, 1).contiguous()

In [ ]:
if CFG.get('task') == 'canonical_reconstruction':
    # identical template, distinct condition IDs for training/validation/evaluation rollout roles
    canonical = render(images, 0, 0, 0, 1.0, [1.0, 1.0, 1.0])
    rows = [{'condition': i, 'split': split, 'angle': 0, 'dx': 0, 'dy': 0, 'saturation': 1.0,
             'red_gain': 1.0, 'green_gain': 1.0, 'blue_gain': 1.0}
            for i, split in enumerate(['train', 'validation', 'test_reconstruction'])]
    condition_df = pd.DataFrame(rows)
    split_ids = {row['split']: [row['condition']] for row in rows}
    PACK = {'targets': canonical[None].expand(3, -1, -1, -1, -1).contiguous().half(),
            'cue_values': torch.tensor([[1.0, 0.0, 1.0, 1.0, 1.0, 1.0]]).expand(3, -1).contiguous(),
            'coordinates': torch.tensor([[CFG['grid'] // 2, CFG['grid'] // 2]]).expand(3, -1).contiguous(),
            'splits': split_ids}
    torch.save(PACK, OUTPUT / 'dataset.pt')
    condition_df.to_csv(OUTPUT / 'conditions.csv', index=False)
    display(condition_df)
else:
    # disjoint composition splits; novel angles are never used for fitting or checkpoint selection
    shift = CFG['shift']
    positions = [-shift, 0, shift]
    palettes = [(1.0, 1.0, 1.0), (0.8, 1.0, 0.75)]
    seen = list(product([0, 90, 180, 270], positions, positions, [0.0, 0.5, 1.0], palettes))
    order = rng.permutation(len(seen))
    train_end, val_end = int(0.7 * len(seen)), int(0.85 * len(seen))
    split_by_index = {int(index): ('train' if rank < train_end else 'validation' if rank < val_end else 'test_composition')
                      for rank, index in enumerate(order)}
    novel_positions = [-max(1, shift - 1), max(1, shift - 1)]
    angle_only = list(product([45, 135, 225, 315], positions, positions, [0.0, 0.5, 1.0], palettes))
    color_only = list(product([0, 90, 180, 270], positions, positions, [0.25, 0.75], palettes))
    translation_only = list(product([0, 90, 180, 270], novel_positions, novel_positions, [0.0, 0.5, 1.0], palettes))
    novel = list(product([45, 135, 225, 315], novel_positions, novel_positions, [0.25, 0.75], palettes))
    rows, targets, cues, coordinates = [], [], [], []
    for split, cases in [('seen', seen), ('test_angle', angle_only), ('test_color', color_only),
                          ('test_translation', translation_only), ('test_joint', novel)]:
        for index, (angle, dx, dy, saturation, gains) in enumerate(cases):
            this_split = split_by_index[index] if split == 'seen' else split
            theta = np.deg2rad(angle)
            rows.append({'condition': len(rows), 'split': this_split, 'angle': angle, 'dx': dx, 'dy': dy,
                         'saturation': saturation, 'red_gain': gains[0], 'green_gain': gains[1], 'blue_gain': gains[2]})
            targets.append(render(images, angle, dx, dy, saturation, gains))
            cues.append([float(np.cos(theta)), float(np.sin(theta)), saturation, *gains])
            coordinates.append([CFG['grid'] // 2 + dy, CFG['grid'] // 2 + dx])
    condition_df = pd.DataFrame(rows)
    split_ids = {name: condition_df.loc[condition_df['split'].eq(name), 'condition'].tolist()
                 for name in ['train', 'validation', 'test_composition', 'test_angle', 'test_color', 'test_translation', 'test_joint']}
    PACK = {'targets': torch.stack(targets).half(), 'cue_values': torch.tensor(cues, dtype=torch.float32),
            'coordinates': torch.tensor(coordinates, dtype=torch.long), 'splits': split_ids}
    torch.save(PACK, OUTPUT / 'dataset.pt')
    condition_df.to_csv(OUTPUT / 'conditions.csv', index=False)
    display(condition_df.groupby('split', sort=False).agg(n=('condition', 'size'), angles=('angle', lambda x: sorted(x.unique()))).reset_index())

In [ ]:
# preparation establishes the template/split contract once
if CFG.get('task') == 'canonical_reconstruction':
    assert torch.equal(PACK['targets'][0], PACK['targets'][1]) and torch.equal(PACK['targets'][1], PACK['targets'][2])
    alpha = PACK['targets'][0, 0, ..., 3]
    assert max(alpha[0].max(), alpha[-1].max(), alpha[:, 0].max(), alpha[:, -1].max()) < 0.01
    assert len(set(sum(split_ids.values(), []))) == 3
    display(pd.DataFrame([{'check': 'same template, distinct rollout roles, unclipped target', 'passed': True}]))
else:
    # split/transform checks occur at preparation only
    target_alpha = PACK['targets'][..., 3].float()
    edge_alpha = torch.cat([target_alpha[..., 0, :].flatten(), target_alpha[..., -1, :].flatten(),
                            target_alpha[..., :, 0].flatten(), target_alpha[..., :, -1].flatten()])
    assert edge_alpha.max().item() < 0.01, 'a transformed target touches the boundary'
    all_ids = sum(split_ids.values(), [])
    assert len(all_ids) == len(set(all_ids)), 'split overlap'
    assert set(condition_df.query("split == 'train'")['angle']).isdisjoint(set(condition_df.query("split == 'test_angle'")['angle']))
    grey_ids = condition_df.loc[condition_df['saturation'].eq(0), 'condition'].tolist()
    # colored illumination gains intentionally tint greyscale; neutral-gain greyscale is channel-identical
    neutral_grey = condition_df.query('saturation == 0 and red_gain == 1').index.tolist()
    grey = PACK['targets'][neutral_grey, ..., :3].float()
    assert (grey[..., 0] - grey[..., 1]).abs().max() < 0.002
    display(pd.DataFrame([{'check': 'split separation, novel angles, unclipped targets, greyscale', 'passed': True}]))

In [ ]:
# fixed-template preview is labeled as reconstruction rather than held-out generalization
if CFG.get('task') == 'canonical_reconstruction':
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='white', palette='colorblind')
    rgba = PACK['targets'][0, 0].float().numpy()
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(rgba[..., :3] + 1 - rgba[..., 3:4])
    ax.set_title('single fixed training target')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'target_examples.png', dpi=160)
    plt.show()
else:
    # target contact sheet
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='white', palette='colorblind')
    sample_ids = [split_ids['train'][0], split_ids['validation'][0], split_ids['test_composition'][0], split_ids['test_angle'][0]]
    fig, axes = plt.subplots(2, 4, figsize=(9, 5))
    for column, condition in enumerate(sample_ids):
        for label in range(2):
            rgba = PACK['targets'][condition, label].float().numpy()
            axes[label, column].imshow(rgba[..., :3] + 1 - rgba[..., 3:4])
            axes[label, column].set_title(f"{condition_df.loc[condition, 'split']} / {['cat', 'smiley'][label]}", fontsize=9)
            axes[label, column].axis('off')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'target_examples.png', dpi=160)
    plt.show()

In [ ]:
# conditioning preflight; independent table-to-tensor alignment, all conditions
if CFG.get('conditioning_diagnostic', False):
    theta = torch.tensor(np.deg2rad(condition_df['angle'].to_numpy().copy()), dtype=torch.float32)
    expected = torch.column_stack([theta.cos(), theta.sin(), torch.tensor(condition_df[['saturation', 'red_gain', 'green_gain', 'blue_gain']].to_numpy().copy(), dtype=torch.float32)])
    assert torch.allclose(PACK['cue_values'], expected, atol=1e-6, rtol=0), 'condition/cue mapping'
    xy = torch.tensor(condition_df[['dy', 'dx']].to_numpy().copy()) + CFG['grid'] // 2
    assert torch.equal(PACK['coordinates'], xy), 'condition/seed location mapping'
    for row in condition_df.itertuples():
        expected_target = render(images, row.angle, row.dx, row.dy, row.saturation, [row.red_gain, row.green_gain, row.blue_gain]).half()
        assert torch.equal(PACK['targets'][row.condition], expected_target), 'condition/target mapping'
    checks = pd.DataFrame([{'check': name, 'passed': True, 'conditions': len(condition_df)} for name in
                           ['cue encoding matches condition table', 'seed locations match translations', 'targets match condition table']])
    checks.to_csv(OUTPUT / 'conditioning_preflight.csv', index=False)
    display(checks)
